# Модуль Г

Сначала доставьте и разгрузите складскую партию, затем перевезите оставшиеся контейнеры в карантин. Отдельно подготовьте обработку температуры, скорости и ошибок камеры.

Описание решения (не более 1000 символов)
Поезд ведёт фоновый контроллер по цветовым меткам (вероятностный фильтр; нумерация меток и полярность определяются по стартовой метке). Кран грузит три нужных контейнера на состав, три лишних в буфер; Диспетчер оповещается в каждой зоне. Склад: робот 2 снимает контейнеры магнитом и оставляет за линией; затем состав возвращается, кран переносит буфер, робот 1 разгружает карантин. После возвращения робота 1 попытка завершается. CybP_01: битый кадр отбраковывается, warning с frameId, уведомление dispatcher и crane.1, повтор после cooldown, регистрация по целому кадру. CybP_02: 68 C - 0,075 м/с (без нагрева), 80 C - 0,05, после 30 C - 0,1; intent в журнале, уведомление dispatcher и train.1.

In [ ]:
import time, itertools
import numpy as np
import cv2
from quasar_client import QuasarClient, QuasarCommandError, QuasarTimeoutError, QuasarRunStoppedError, QuasarProtocolError
client = QuasarClient.from_environment(timeout=5)
client.wait_for_simulation_start()
client.configure_incidents(['CybP_01', 'CybP_02'])
client.start_attempt()
T0 = time.monotonic()
def ts(): return round(time.monotonic() - T0, 1)
def log(level, msg, **d):
    print(f'[{ts()}] {level}: {msg}', d if d else '')
    try: client.append_log(level, 'MOD_G', msg[:300], d or None)
    except Exception: pass
def _gen(n=686, w=6):
    s, used, nxt = [], set(), [0]
    while len(s) < n:
        k, c, placed = len(s), nxt[-1], False
        while c < 4:
            if not s or c != s[-1]:
                if k >= w - 1:
                    win = tuple(s[k-w+1:]) + (c,)
                    if win in used: c += 1; continue
                    used.add(win)
                s.append(c); nxt[-1] = c + 1; nxt.append(0); placed = True; break
            c += 1
        if not placed:
            nxt.pop(); last = s.pop(); k = len(s)
            if k >= w - 1: used.discard(tuple(s[k-w+1:]) + (last,))
    return s
SEQ = np.array(_gen()); N = len(SEQ)
G = (-1, 1)
def classify(rgb):
    ch = [rgb['r'], rgb['g'], rgb['b']]; v = max(ch)
    if v < 45: return 0
    if sorted(ch)[1] > 0.6 * v: return None
    return 1 + ch.index(v)
def _shift(a, k):
    out = np.zeros_like(a)
    if k == 0: return a.copy()
    if k > 0: out[k:] = a[:N-k]; out[N-1] += a[N-k:].sum()
    else: out[:N+k] = a[-k:]; out[0] += a[:-k].sum()
    return out
class Loc:
    def __init__(self):
        self.b = np.full((2, N), 0.5 / N); self.t = None; self.v = 0.0
    def predict(self, now):
        if self.t is not None and self.v != 0.0:
            for h in (0, 1):
                d = G[h] * self.v * (now - self.t) / 0.01
                n = int(np.floor(d)); f = d - n
                self.b[h] = (1 - f) * _shift(self.b[h], n) + f * _shift(self.b[h], n + 1)
        for h in (0, 1):
            x = self.b[h]; nb = 0.76 * x
            nb[1:] += 0.12 * x[:-1]; nb[:-1] += 0.12 * x[1:]; nb[0] += 0.12 * x[0]; nb[-1] += 0.12 * x[-1]
            self.b[h] = nb
        self.t = now
    def update(self, obs):
        if obs is None: return
        nb = self.b * np.where(SEQ == obs, 0.8, 0.05)[None, :]; s = nb.sum()
        self.b = nb / s if s > 1e-300 else np.full((2, N), 0.5 / N)
    def est(self):
        m = self.b.sum(axis=0); k = int(np.argmax(m))
        s0, s1 = self.b[0].sum(), self.b[1].sum()
        return k, float(m[max(0, k-2):k+3].sum()), (0 if s0 >= s1 else 1), float(max(s0, s1))
class Train:
    def __init__(self, send, v=0.1, vm=0.05, vf=0.02, far=60, fine=14):
        self.send, self.v, self.vm, self.vf, self.far, self.fine = send, v, vm, vf, far, fine
        self.f = Loc(); self.cmd = 0.0; self.mode = 'idle'; self.est = None; self.target = None
        self.t_mode = 0.0; self.blind = 0.0; self.cap = None; self.ann = True; self.pm = None; self.dist = 0.0
    def vv(self): return min(self.v, self.cap) if self.cap else self.v
    def _set(self, s, now):
        self.f.predict(now); self.f.v = s; self.cmd = s; self.send(s)
    def start(self, target, now):
        self.target = target; self.mode = 'run'; self.ann = False; self.f.t = now; self.blind = now; self._set(self.vv(), now)
    def step(self, rgb, now):
        f = self.f; f.predict(now); f.update(classify(rgb)); k, c, h, ph = f.est()
        loc = c >= 0.8 and ph >= 0.95
        if self.pm is None and loc:
            k0 = k - G[h] * self.dist * 100; dn, dp = abs(k0 - (218, 468)[h]), abs(k0 - (241, 491)[h])
            self.pm = 0 if dn < dp and dn < 12 else 1
            print('pm', self.pm, round(k0))
        P = self.pm == 1
        q = 709.5 - k if P else k; self.est = round(q)
        s = G[h] * (-1 if P else 1)
        if self.mode in ('idle', 'done'): return
        if self.mode == 'settle':
            if now - self.t_mode >= 0.8: self.mode = 'run' if (loc and abs(q - self.target) >= 1) else 'done'
            return
        if self.mode == 'rev':
            if now - self.t_mode < 0.3: return
            self.mode = 'run'
        if loc:
            self.blind = now; err = self.target - q
            if self.cmd != 0 and ((s * self.cmd < 0 and q <= LIM_LO) or (s * self.cmd > 0 and q >= LIM_HI)):
                self._set(0.0, now); self.mode = 'settle'; self.t_mode = now; return
            if abs(err) < 1:
                self._set(0.0, now); self.mode = 'settle'; self.t_mode = now; return
            need = 1 if err > 0 else -1
            a = abs(err); want = need * s * (self.vv() if a > self.far else min(self.vm, self.vv()) if a > self.fine else min(self.vf, self.vv()))
            if want != self.cmd:
                if self.cmd != 0 and (want > 0) != (self.cmd > 0):
                    self._set(0.0, now); self.mode = 'rev'; self.t_mode = now; return
                self._set(want, now)
        else:
            if self.cmd != 0 and now - self.blind > 8: self._set(0.0, now); self.blind = now
            elif self.cmd == 0 and now - self.blind > 1.0: self._set(self.vv(), now)
def send_speed(s):
    client.execute_gcode('train.1', 'M121' if s == 0 else f'M120 S{s:.2f}', wait=False)
LIM_LO, LIM_HI = 36, 650
STORE = 42
TRAIN_HOME = 336
QUAR = 643
tr = Train(send_speed)
LAST = {}
def notify(ids, text):
    try: client.notify_nodes(ids, text[:200])
    except Exception as e: log('warning', f'notify: {e}'[:200])
def arrived(target, k):
    zone = {STORE: 'storage zone', QUAR: 'quarantine zone'}.get(target, 'unloading zone')
    log('info', f'поезд на месте, готов: {zone}', block=k, t=ts())
    notify(['dispatcher'], f'train.1 ready: {zone}, track block {k}')
def cyp(intent, T, msg):
    d = {'intent': intent, 'temperatureC': round(T, 1), 'limitMps': tr.cap or tr.v}
    print(f'[{ts()}] CybP_02 {intent}: {msg} T={T:.1f}')
    try: client.append_log('info', 'CybP_02', f'CybP_02 низкий уровень: {msg}'[:300], d)
    except Exception as e: print('append_log', e)
    notify(['dispatcher', 'train.1'], f'CybP_02: {intent}, motor {T:.0f}C, {msg}'[:190])
def therm(st):
    T = st['cyber']['train']['motorTemperatureC']
    if T >= 80 and tr.cap != 0.05: tr.cap = 0.05; cyp('brake', T, 'близко к критическому порогу 85C, скорость до 0.05 м/с')
    elif T >= 68 and tr.cap is None: tr.cap = 0.075; cyp('brake', T, 'превышен средний порог 65C, скорость снижена до 0.075 м/с (без нагрева)')
    elif T < 70 and tr.cap == 0.05: tr.cap = 0.075; cyp('accelerate', T, 'температура снизилась, скорость до 0.075 м/с')
    elif T <= 30 and tr.cap is not None: tr.cap = None; cyp('accelerate', T, 'двигатели остыли, разгон до 0.1 м/с')
def pump():
    w = LAST.get('p', 0) + 0.05 - time.monotonic()
    if w > 0: time.sleep(w)
    LAST['p'] = time.monotonic()
    client.execute_gcode('train.1', 'M150', wait=False)
    st = client.get_state(); LAST['st'] = st; n = time.monotonic()
    if tr.pm is None: tr.dist += abs(st['cyber']['train']['measuredSpeedMetersPerSecond']) * (n - LAST.get('pt', n))
    LAST['pt'] = n
    therm(st)
    if tr.mode == 'done' and not tr.ann: tr.ann = True; arrived(tr.target, tr.est)
    try: tr.step(st['train']['gcode']['lastResponse']['rgb'], time.monotonic())
    except (KeyError, TypeError): pass
    return st
def nap(sec):
    t = time.monotonic()
    while time.monotonic() - t < sec: pump(); time.sleep(0.03)
MVXY, FXY = 'G1', 9000
MVZ, FZU, FZD = 'G1', 7200, 7200
ZT, ZW = 80, 170
ZC = [None]
BUF_X = 540
BUF_Y = (130, 250, 370, 60, 440)
def crane(prog, tmo=40, tag=''):
    client.execute_gcode('crane.1', prog, wait=False)
    t0 = time.monotonic()
    while True:
        st = pump(); g = st['crane']['gcode']
        if g.get('lastError'): raise QuasarCommandError(str(g['lastError']))
        if g.get('busy') is False:
            return st
        if time.monotonic() - t0 > tmo: raise QuasarTimeoutError('crane timeout')
        time.sleep(0.03)
def clamp(X, Y): return min(max(X, 0), 1106), min(max(Y, 0), 500)
def crane_xy(st): return st['crane']['portal'] * 1000, st['crane']['cart'] * 1000
def mxy(X, Y, Z=None):
    X, Y = clamp(X, Y)
    return f'{MVXY} X{X:.0f} Y{Y:.0f}' + ('' if Z is None else f' Z{Z}') + f' F{FXY}'
def zup(Z): return f'{MVZ} Z{Z} F{FZU}'
def zdn(Z): return f'G1 Z{Z} F{FZD}'
ARUCO = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
CAM = {'last': -10.0, 'hold': 0.0, 'ord': 0}
def cyb_bad(fid, ep, why):
    log('warning', f'ПРЕДУПРЕЖДЕНИЕ CybP_01: нарушена целостность кадра {fid}, кадр отбракован'[:290], t=ts())
    try: client.append_log('warning', 'CybP_01', f'CybP_01 средний уровень, ПРЕДУПРЕЖДЕНИЕ: сбой зрения, кадр {fid} повреждён и отбракован; повтор запроса после cooldown, регистрация только по кадру ok',
                           {'level': 'средний', 'frameId': fid, 'episodeId': ep or '', 'reason': why[:100]})
    except Exception as e: print('append_log', e)
    notify(['dispatcher', 'crane.1'], f'CybP_01: frame {fid} corrupted, rejected, retry after cooldown')
    CAM['shown'] = True
def cam_try():
    w = max(CAM['last'] + 1.6, CAM['hold']) - time.monotonic()
    if w > 0: nap(w)
    try: o = client.get_crane_camera_observation()
    except QuasarCommandError as e:
        CAM['last'] = time.monotonic()
        if 'cooldown' in str(e): return None
        raise
    except QuasarProtocolError as e:
        CAM['last'] = time.monotonic(); CAM['ord'] += 1
        if 'metadata' not in str(e): raise
        cyb_bad(f"crane-frame-{CAM['ord']}", None, 'invalid-metadata'); return None
    CAM['last'] = time.monotonic(); f = o.fault; ig = f['integrity']; CAM['ord'] = f['captureOrdinal']
    diff = sum(a != b for a, b in zip(ig['expected'], ig['observed']))
    if ig['status'] != 'ok' or diff:
        if f['episodeId'] == 'CybP_01.1' and not CAM['hold']: CAM['hold'] = time.monotonic() + 10.6
        cyb_bad(o.frame_id, f['episodeId'], f"status={ig['status']} diff={diff}"); return None
    return o.frame, o.frame_id
def capture_ok():
    for _ in range(40):
        r = cam_try()
        if r: return r
    raise RuntimeError('камера: нет целого кадра')
IDS = {30, 32, 33, 34, 35, 36}
PX = 1.5
def detect_markers():
    frame, fid = capture_ok()
    cs, ids, _ = cv2.aruco.detectMarkers(frame, ARUCO)
    out = {}
    if ids is not None:
        for cc, i in zip(cs, ids.ravel()):
            if int(i) in IDS: out[int(i)] = (float(cc[0][:, 0].mean()), float(cc[0][:, 1].mean()))
    return out, fid
def scan_at(Y, X=933):
    st = crane(mxy(X, Y, 0), tag='scan')
    Xc, Yc = crane_xy(st)
    mk, fid = detect_markers()
    return {i: (Xc + (320 - cx) / PX, Yc + (240 - cy) / PX, fid) for i, (cx, cy) in mk.items()}
BIAS = [-5.0, 0.0]
def try_grab(X, Y):
    X, Y = clamp(X, Y)
    zs = [100 + 5 * k for k in range(19)] if ZC[0] is None else [ZC[0] + ZADD[0] + d for d in ((-10, -5, 0, 5, 10, 15, 25, 35) if ZADD[0] else (0, 5, 10, 20, 35, 50))]
    zs = [z for z in zs if z <= 240]
    try:
        crane([mxy(X, Y, ZT), zdn(zs[0] if ZC[0] is not None else 100)], tag='grab')
        for z in zs:
            try:
                crane([zdn(z), 'M106 P1 S255'], tag='grab')
                if ZC[0] is None: log('info', f'высота захвата найдена: Z={z}')
                ZC[0] = z if ZC[0] is None else min(ZC[0], z)
                return True
            except QuasarCommandError as e:
                if 'blocked' in str(e) or 'limit' in str(e): raise
                crane(['M106 P1 S0'], tag='grab')
        raise QuasarCommandError('magnet-capture-failed на всех высотах')
    except (QuasarCommandError, QuasarTimeoutError) as e:
        st = client.get_state()
        log('warning', f'захват не удался ({X:.0f},{Y:.0f}): {e}', cx=round(st['crane']['portal'] * 1000),
            cy=round(st['crane']['cart'] * 1000), cz=round(st['crane']['zRod'] * 1000))
        try: crane(['M106 P1 S0', zup(ZT)], tag='grab')
        except Exception: pass
        return False
def grab(i, pos):
    X, Y = pos[i]
    if try_grab(X + BIAS[0], Y + BIAS[1]): return True
    for dx, dy in ((-4, 0), (4, 0), (0, 5), (0, -5), (-8, 0), (8, 0)):
        if try_grab(X + BIAS[0] + dx, Y + BIAS[1] + dy):
            BIAS[0] += dx; BIAS[1] += dy; log('info', f'захват {i} со смещением', bx=BIAS[0], by=BIAS[1]); return True
    return False
def to_buffer(k):
    try: crane([zup(ZT), mxy(BUF_X, BUF_Y[k]), zdn((ZC[0] or 130) - 10)], tag='buf')
    except QuasarCommandError as e:
        if 'blocked' not in str(e): raise
    crane(['M106 P1 S0', 'G4 P100', zup(ZT)], tag='buf')
WAGON_X, BAY0_Y, BAY_PITCH = 110, 233, 91
OFFS = (0, 3, -3, 6, -6, 9, -9, 12, -12, 15, -15, 18, -18, 21, -21, 25, -25, 29, -29)
def to_wagon(bay_y):
    for d in OFFS:
        y = bay_y + d
        try:
            crane([zup(ZT), mxy(WAGON_X, y), zdn(ZW)], tag='wag')
        except QuasarCommandError as e:
            if 'blocked' not in str(e): raise
            continue
        crane(['M106 P1 S0', 'G4 P100', zup(ZT)], tag='wag')
        log('info', f'контейнер на вагон: Y={y}', onTrain=client.get_state()['containers']['onTrain'])
        return y
    crane(zup(ZT), tag='wag')
    return None
def agg(items):
    out = {}
    for it in items: out[it['nomenclature']] = out.get(it['nomenclature'], 0) + it['massKg']
    return out
DX = (0, -0.02, 0.02, -0.035, 0.035, 0.05, -0.05)
def rg(r, prog, wait=True):
    try: return client.execute_gcode(r, prog, wait=wait, timeout=30)
    except QuasarRunStoppedError: raise
    except Exception as e: print('gcode', r, repr(e)[:120]); return None
def rpos(r):
    q = next(q for q in client.get_state()['robots'] if q['id'] == r)
    return q['x'], q['y']
def pick(r, s, y0, A, xg):
    for dx in DX:
        x = round(xg + dx, 3)
        rg(r, f'G90\nG1 X{x} Y{y0 + 0.52 * s:.3f} A{A} F0.2')
        rg(r, f'M120 X0.05 Y0 A{A}', wait=False)
        t0 = time.monotonic(); got = None
        while time.monotonic() - t0 < 6:
            py = rpos(r)[1]
            if (py - y0) * s > 0.63: break
            if client.set_robot_magnet(r, True).get('accepted'): got = (x, py); break
            time.sleep(0.04)
        try: client.execute_gcode(r, 'M121', wait=True, timeout=10)
        except QuasarRunStoppedError: raise
        except Exception: pass
        if got: return got
        log('warning', f'{r}: промах на x={x}')
    return None
def unload(r, s, y0, A, nom, drops):
    sh, k = 0.0, 0
    for rnd in range(2):
        for xg in nom:
            if client.get_state()['containers']['onTrain'] == 0 or k >= 3: return
            got = pick(r, s, y0, A, xg + sh)
            if not got: continue
            sh = got[0] - xg; xd = drops[k]; k += 1; yr = y0 + 0.25 * s
            rg(r, f'G90\nG1 X{got[0]} Y{yr:.3f} A{A} F0.2\nG1 X{xd} Y{yr:.3f} A{A} F0.2')
            client.set_robot_magnet(r, False); time.sleep(0.2)
            rg(r, f'G1 X{xd} Y{y0} A{A} F0.2')
            log('info', f'{r}: контейнер {k} за линией', x=xd, onTrain=client.get_state()['containers']['onTrain'])
TLIM = 200
TDELIV = 300
ZADD = [0]
completed = False
try:
    log('info', 'старт попытки модуля Г')
    tr.start(TRAIN_HOME, time.monotonic())
    cam_try()
    crane(['G90', mxy(933, 200, 0)], tag='pre')
    while True:
        st = pump()
        if (st.get('moduleA') or {}).get('shipPhase') == 'ready': break
        time.sleep(0.03)
    log('info', 'корабль готов к разгрузке', t=ts())
    pos, FID = {}, {}
    for Y in (200, 420, 120, 480):
        for i, (x, y, fid) in scan_at(Y).items():
            if i not in pos: pos[i] = (x, y); FID[i] = fid
        if len(pos) == 6: break
    log('info', 'маркеры: ' + ' '.join(f'{i}:({p[0]:.0f},{p[1]:.0f})' for i, p in sorted(pos.items())))
    if len(pos) < 6: raise RuntimeError('не найдены все шесть маркеров')
    request = agg(client.get_cargo_request()['items'])
    info = {}
    for i in sorted(pos):
        info[i] = client.get_container_cargo(i)['items']
        resp = client.register_container_cargo(i, info[i], frame_id=FID[i])
        if CAM.get('shown') and i == min(pos):
            try: client.append_log('info', 'CybP_01', f'CybP_01 средний уровень: регистрация по целому кадру {FID[i]} (status ok)', {'level': 'средний', 'frameId': FID[i]})
            except Exception as e: print('append_log', e)
        if resp.get('matchesExpected') is not True: log('error', f'сверка контейнера {i} не прошла: {resp}'[:290])
    need = next((c for c in itertools.combinations(sorted(pos), 3)
                 if agg([it for i in c for it in info[i]]) == request), None)
    if need is None: raise RuntimeError('нет тройки контейнеров под заявку')
    rest = [i for i in sorted(pos) if i not in need]
    log('info', f'зарегистрированы все 6; требуются {list(need)}, в буфер {rest}', t=ts())
    done, nbuf, ys, bufk = set(), [0], [], {}
    def buffer_pass():
        for i in sorted([j for j in rest if j not in done], key=lambda j: pos[j][1]):
            if time.monotonic() - T0 > TLIM: return
            if grab(i, pos):
                kb = min(nbuf[0], 4); to_buffer(kb); nbuf[0] += 1; done.add(i); bufk[i] = kb
    def to_train(lst, src, ys, lim):
        for k, i in enumerate(lst):
            if i in done or time.monotonic() - T0 > lim: continue
            if k == 0 or not ys: guess = BAY0_Y
            elif k == 1: guess = ys[0] + BAY_PITCH
            else: guess = ys[0] - BAY_PITCH
            ZADD[0] = 100 if i in bufk else 0
            ok = grab(i, src)
            ZADD[0] = 0
            if not ok: continue
            y = to_wagon(guess)
            if y is None:
                log('error', f'контейнер {i} не встал в карман, сбрасываем в буфер')
                kb = min(nbuf[0], 4); to_buffer(kb); nbuf[0] += 1; bufk[i] = kb
                src[i] = (BUF_X - BIAS[0], BUF_Y[kb] - BIAS[1])
                continue
            done.add(i); ys.append(y)
    buffer_pass()
    while tr.mode != 'done' and time.monotonic() - T0 < TLIM:
        pump(); time.sleep(0.03)
    to_train(need, pos, ys, TLIM)
    for _ in range(2):
        if len(done) == 6 or time.monotonic() - T0 > TLIM: break
        log('warning', f'повторный проход, осталось {sorted(set(pos) - done)}')
        buffer_pass(); to_train([i for i in need if i not in done], pos, ys, TLIM)
    cs = client.get_state()['containers']
    loaded = cs['onTrain'] == 3 and cs['onShip'] == 0
    crane(zup(0), tag='home')
    if loaded:
        notify(['dispatcher'], 'ship unloading complete: 3 requested containers on train')
        tr.start(STORE, time.monotonic()); log('info', 'груз на составе, поезд идёт в зону хранения', t=ts())
    crane(mxy(0, 0, 0), tag='home')
    while loaded and tr.mode != 'done' and time.monotonic() - T0 < TDELIV:
        pump(); time.sleep(0.03)
    if loaded: log('info' if tr.mode == 'done' else 'error', f'поезд у зоны хранения: блок {tr.est}, цель {STORE}', t=ts())
    unload('robot.2', 1, 2.2, 1.5708, (3.05, 2.955, 2.864), (3.38, 3.12, 2.62))
    notify(['dispatcher'], 'robot.2: storage unloading complete')
    log('info', 'склад разгружен', t=ts(), onTrain=client.get_state()['containers']['onTrain'])
    rg('robot.2', 'G90\nG1 X3.0 Y2.2 A1.5708 F0.2', wait=False)
    tr.start(TRAIN_HOME, time.monotonic())
    while tr.mode != 'done' and time.monotonic() - T0 < 420:
        pump(); time.sleep(0.03)
    pos2 = {i: ((BUF_X - BIAS[0], BUF_Y[bufk[i]] - BIAS[1]) if i in bufk else pos[i]) for i in rest}
    done.clear(); ys2 = []
    rest2 = sorted(rest, key=lambda i: pos2[i][1])
    to_train(rest2, pos2, ys2, 440)
    for _ in range(2):
        if len(done) == len(rest) or time.monotonic() - T0 > 440: break
        to_train([i for i in rest2 if i not in done], pos2, ys2, 440)
    crane(zup(0), tag='home'); crane(mxy(0, 0, 0), tag='home')
    if client.get_state()['containers']['onTrain'] > 0:
        tr.start(QUAR, time.monotonic()); log('info', 'карантинная партия на составе, поезд идёт в карантин', t=ts())
        while tr.mode != 'done' and time.monotonic() - T0 < 480:
            pump(); time.sleep(0.03)
        log('info' if tr.mode == 'done' else 'error', f'поезд у карантина: блок {tr.est}, цель {QUAR}', t=ts())
        unload('robot.1', -1, 1.4, -1.5708, (3.12, 3.03, 2.935), (3.40, 3.22, 2.62))
        notify(['dispatcher'], 'robot.1: quarantine unloading complete')
        rg('robot.1', 'G90\nG1 X3.0 Y1.4 A-1.5708 F0.2')
    completed = True
    log('info', 'задачи 4-6 выполнены, завершаем попытку', t=ts(), containers=client.get_state()['containers'])
    try: client.stop_attempt()
    except QuasarRunStoppedError: pass
except QuasarRunStoppedError:
    log('info' if completed else 'error', 'прогон остановлен' + ('' if completed else ' до завершения'))
except Exception as e:
    log('error', f'сбой сценария: {e!r}')
    raise
finally:
    try: send_speed(0)
    except Exception: pass
